In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import os
os.chdir('/content/drive/MyDrive/PhD/newtests-v-1.5')

In [4]:
import numpy as np
import pandas as pd
import copy

import torch
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import MinMaxScaler

from data import fetch_dataset
from util import move_sliding_window, num_params
from model import LSTMModel
from algorithm import fedavg

pd.set_option('display.max_columns', None)
pd.set_option('display.max_columns', None)
np.set_printoptions(suppress=True, floatmode='fixed')

In [5]:
if torch.cuda.is_available():
    device = torch.device("cuda")
    print('running on gpu')
else:
    device = torch.device("cpu")

running on gpu


# Parameters

In [6]:
window_size = 90 # Define window_size period and split inputs/labels\
batch_size = 1024
label_col_index = 0

# seq_len = 90  # (timestamps)
hidden_dim = 50
n_layers = 2
lr = 0.0001
output_dim = 1

#fed train params
num_local_epochs = 1
max_rounds = 100 #nb of total rounds for training


# Data Preperation

In [7]:
dataframes = fetch_dataset("./cleaned")

Floor0
Floor1
Floor10
Floor11
Floor12
Floor13
Floor14
Floor15
Floor16
Floor2
Floor20
Floor21
Floor22
Floor23
Floor24
Floor25
Floor26
Floor3
Floor4
Floor5
Floor6
Floor17_p10
Floor18_p7
Floor19_p8
Floor27_p2
Floor28_p4
Floor29_p3
Floor7_p6
Floor8_p6
Floor9_p7


In [8]:
n_clients = len(dataframes) #total number of clients to partition data into

In [9]:
for _, df in dataframes.items():
    print(df.columns)
    print(df.shape)

Index(['total_demand', 'AC', 'Light', 'Plug', 'Year', 'Month', 'Day', 'Hour',
       'Minute'],
      dtype='object')
(72864, 9)
Index(['total_demand', 'AC', 'Light', 'Plug', 'Year', 'Month', 'Day', 'Hour',
       'Minute'],
      dtype='object')
(72864, 9)
Index(['total_demand', 'AC', 'Light', 'Plug', 'Year', 'Month', 'Day', 'Hour',
       'Minute'],
      dtype='object')
(72864, 9)
Index(['total_demand', 'AC', 'Light', 'Plug', 'Year', 'Month', 'Day', 'Hour',
       'Minute'],
      dtype='object')
(72864, 9)
Index(['total_demand', 'AC', 'Light', 'Plug', 'Year', 'Month', 'Day', 'Hour',
       'Minute'],
      dtype='object')
(72864, 9)
Index(['total_demand', 'AC', 'Light', 'Plug', 'Year', 'Month', 'Day', 'Hour',
       'Minute'],
      dtype='object')
(72864, 9)
Index(['total_demand', 'AC', 'Light', 'Plug', 'Year', 'Month', 'Day', 'Hour',
       'Minute'],
      dtype='object')
(72864, 9)
Index(['total_demand', 'AC', 'Light', 'Plug', 'Year', 'Month', 'Day', 'Hour',
       'Minute'],
 

# Build the training set

In [10]:
train_loader = []
test_loader = []
label_scalers = []
for _, df in dataframes.items():
    inputs_cols_indices = range(0, df.shape[1])  # use (total_demand,Year,Month,Day,Hour,Minute) columns as features
    #move the window
    inputs, labels = move_sliding_window(
        df.values,
        window_size,
        inputs_cols_indices=inputs_cols_indices,
        label_col_index=label_col_index
    )

    # Normalize the input data columns
    sc = MinMaxScaler()
    # Obtaining the scaler for the labels(usage data) so that output can be re-scaled to actual value during evaluation
    label_sc = MinMaxScaler()

    # Split data into train/test portions and combining all data into a single array
    test_portion = int(0.2 * len(inputs))

    train_x = sc.fit_transform(inputs[:-test_portion].reshape(-1, window_size * df.shape[1]))
    train_x = train_x.reshape(-1, window_size, df.shape[1])
    train_y = label_sc.fit_transform(labels[:-test_portion])

    test_x = sc.transform(inputs[-test_portion:].reshape(-1, window_size * df.shape[1]))
    test_x = test_x.reshape(-1, window_size, df.shape[1])
    test_y = label_sc.transform(labels[-test_portion:])

    # test_x.append(testx)
    # test_y.append(testy)
    label_scalers.append(label_sc)

    # pytorch data loaders
    train_data = TensorDataset(torch.from_numpy(train_x).to(device), torch.from_numpy(train_y).to(device))
    train_loader.append(DataLoader(train_data, batch_size=batch_size, drop_last=True))# Drop the last incomplete batch
    test_data = TensorDataset(torch.from_numpy(test_x).to(device), torch.from_numpy(test_y).to(device))
    test_loader.append(DataLoader(test_data, batch_size=batch_size))# Drop the last incomplete batch

    # release some memory
    del train_x, train_y
input_dim = next(iter(train_loader[0]))[0].shape[2]  # 22

(72774, 90, 9) (72774, 1)
(72774, 90, 9) (72774, 1)
(72774, 90, 9) (72774, 1)
(72774, 90, 9) (72774, 1)
(72774, 90, 9) (72774, 1)
(72774, 90, 9) (72774, 1)
(72774, 90, 9) (72774, 1)
(72774, 90, 9) (72774, 1)
(72774, 90, 9) (72774, 1)
(72774, 90, 9) (72774, 1)
(72774, 90, 9) (72774, 1)
(72774, 90, 9) (72774, 1)
(72774, 90, 9) (72774, 1)
(72774, 90, 9) (72774, 1)
(72774, 90, 9) (72774, 1)
(72774, 90, 9) (72774, 1)
(72774, 90, 9) (72774, 1)
(72774, 90, 9) (72774, 1)
(72774, 90, 9) (72774, 1)
(72774, 90, 9) (72774, 1)
(72774, 90, 9) (72774, 1)
(72774, 90, 9) (72774, 1)
(72774, 90, 9) (72774, 1)
(72774, 90, 9) (72774, 1)
(72774, 90, 9) (72774, 1)
(72774, 90, 9) (72774, 1)
(72774, 90, 9) (72774, 1)
(72774, 90, 9) (72774, 1)
(72774, 90, 9) (72774, 1)
(72774, 90, 9) (72774, 1)


# LSTM model

In [11]:
lstm = LSTMModel(input_dim, hidden_dim, output_dim, n_layers)
model_type = 'LSTM'
print(lstm)
print(num_params(lstm))

LSTMModel(
  (lstm): LSTM(9, 50, num_layers=2, batch_first=True)
  (fc): Linear(in_features=50, out_features=1, bias=True)
  (tanh): Tanh()
)
32651


## FedAvg

In [12]:
%%time
lstm_K5_avg = copy.deepcopy(lstm)
outputs_avg, targets_avg, loss_avg, smape_avg, mae_avg, rmse_avg = fedavg(
    global_model = lstm_K5_avg,
    client_train_loader = train_loader,
    test_loader = test_loader,
    label_sc = label_scalers,
    n_clients = n_clients,
    batch_size = batch_size,
    num_local_epochs = num_local_epochs,
    lr = lr,
    max_rounds = max_rounds,
    model_type = model_type,
    device = device,
)

Streaming output truncated to the last 5000 lines.
11 12
12 13
13 14
14 15
15 16
16 17
17 18
18 19
19 20
20 21
21 22
22 23
23 25
24 26
25 27
26 28
27 29
{0: 0, 1: 0, 2: 1, 3: 0, 5: 0, 6: 0, 7: 0, 8: 0, 9: 0, 10: 1, 11: 2, 12: 1, 13: 2, 14: 2, 15: 0, 16: 0, 17: 0, 18: 0, 19: 0, 20: 0, 21: 0, 22: 0, 23: 0, 25: 0, 26: 1, 27: 0, 28: 0, 29: 0}
Updated clients:  [0, 1, 2, 3, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 25, 26, 27, 28, 29]

starting round 27
clients:  [0, 1, 2, 3, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 25, 26, 27, 28, 29]
round 27, starting client 0/27, id: 0
Epoch [1/1], Train Loss: 0.0033484590593226508
loss error: 0.0371
[0, 1, 3, 4, 5, 6, 7, 8, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 25, 26, 27]
llll
round 27, starting client 1/27, id: 1
Epoch [1/1], Train Loss: 0.003617449845478404
loss error: 0.0525
[0, 1, 3, 4, 5, 6, 7, 8, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 25, 26, 27]
llll
round 27, starting client 2/27, i

OutOfMemoryError: CUDA out of memory. Tried to allocate 392.00 MiB. GPU 0 has a total capacity of 14.75 GiB of which 391.06 MiB is free. Process 117551 has 14.36 GiB memory in use. Of the allocated memory 13.86 GiB is allocated by PyTorch, and 372.97 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [13]:
np.save(f'metrics/fedavg_outputs_C{n_clients}.npy', np.array(outputs_avg, dtype=object))
np.save(f'metrics/fedavg_targets_C{n_clients}.npy', np.array(targets_avg, dtype=object))
np.save(f'metrics/fedavg_loss_C{n_clients}.npy', loss_avg)
np.save(f'metrics/fedavg_smape_C{n_clients}.npy', smape_avg)
np.save(f'metrics/fedavg_mae_C{n_clients}.npy', mae_avg)
np.save(f'metrics/fedavg_rmse_C{n_clients}.npy', rmse_avg)

NameError: name 'outputs_avg' is not defined